In [ ]:
import kagglehub
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1
import pandas as pd
import matplotlib.pyplot as plt #
import seaborn as sns
df = pd.read_csv(path + "/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
print("First 5 rows:")
display(df.head())

In [ ]:
# Task 3
print("\nDataset Info:")
df.info()


In [ ]:
# Task 4
print("\nStatistical Description:") #
display(df.describe())


In [ ]:
# Task 5
plt.figure(figsize=(8,5))
sns.histplot(df["Delivery_Time"], kde=True, bins=30, color="steelblue")
plt.title("Distribution of Delivery Time")
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Task 1
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

df = df.drop(columns=['Order_ID'], errors='ignore')


In [ ]:
# Task 2
print("Missing values per column:")
print(df.isnull().sum())

num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values after imputation:")
print(df.isnull().sum())

In [ ]:
# Task 3
df = df.drop_duplicates()
after = df.shape[0]

print(f"\nDuplicates removed: {after}")


In [ ]:
# Task 4
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [ ]:
# Task 5
from sklearn.preprocessing import StandardScaler

# Features and target
X = df_encoded.drop(columns=["Delivery_Time"])
y = df_encoded["Delivery_Time"]

# Standard scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Task 6
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(X, kde=True, bins=30)
plt.title("Delivery Time Distribution")
plt.show()


In [ ]:
# Task 1
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# Assuming X_scaled and y are already prepared from previous steps

# 1. Define KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

In [ ]:
# Task2
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
models = []  # keep models if you want to reuse the last one for plots

for fold, (train_idx, val_idx) in enumerate(kf.split(X_scaled), 1):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)
    models.append(rf)

    print(f"Fold {fold} | MAE: {mae:.4f}")

print(f"\nAverage MAE across folds: {np.mean(mae_scores):.4f}")


In [ ]:
# Task 1
final_model = models[-1]

importances = final_model.feature_importances_
feature_names = X.columns

feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x=feat_imp.values, y=feat_imp.index, color="steelblue")
plt.title("Feature Importance (RandomForest)")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

In [ ]:
# Task 2
all_preds = []
all_true = []

for train_idx, val_idx in kf.split(X_scaled):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)

    all_preds.extend(y_pred)
    all_true.extend(y_val)

plt.figure(figsize=(8,5))
sns.histplot(all_preds, kde=True, bins=30, color="darkorange")
plt.title("Predicted Delivery Time Distribution")
plt.xlabel("Predicted Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus
from catboost import CatBoostRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores_ensemble = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_scaled), 1):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    cb = CatBoostRegressor(
        depth=6,
        learning_rate=0.1,
        n_estimators=300,
        random_state=42,
        verbose=False
    )

    rf.fit(X_train, y_train)
    cb.fit(X_train, y_train)

    rf_pred = rf.predict(X_val)
    cb_pred = cb.predict(X_val)

    ensemble_pred = (rf_pred + cb_pred) / 2.0

    mae = mean_absolute_error(y_val, ensemble_pred)
    mae_scores_ensemble.append(mae)

    print(f"Fold {fold} | Ensemble MAE: {mae:.4f}")

print(f"\nAverage Ensemble MAE across folds: {np.mean(mae_scores_ensemble):.4f}")